In [30]:
from datasets import load_dataset
import pandas as pd

# Streaming 3,000 rows from the dataset chunks
dataset = load_dataset("winterForestStump/10-K_sec_filings", streaming=True)

collected_rows = []
target_rows = 3000

for chunk_id in dataset.keys():
    if len(collected_rows) >= target_rows:
        break
        
    print(f"Pulling data from chunk: {chunk_id}")

    for row in dataset[chunk_id]:
        collected_rows.append(row)
        if len(collected_rows) == target_rows:
            break

df = pd.DataFrame(collected_rows)


Pulling data from chunk: 001


In [ ]:
import numpy as np

# Filtering out rows where the MD&A section is empty or too short to analyze
mda_column = 'Management’s Discussion and Analysis of Financial Condition and Results of Operations'
df_filtered = df[df[mda_column].fillna("").str.len() > 100].copy()

# Defining a small, distinct list of positive and negative financial words
positive_words = {'growth', 'increase', 'profitable', 'expansion', 'gain', 'success', 'strengthen', 'improvement'}
negative_words = {'loss', 'decline', 'risk', 'deficit', 'decrease', 'adversely', 'litigation', 'failure'}

def financial_labels(text):
    # Convert text to lowercase and split into words
    words = str(text).lower().split()
    
    # Count positive and negative occurrences of the words
    pos_count = sum(1 for word in words if word in positive_words)
    neg_count = sum(1 for word in words if word in negative_words)
    
    # Calculating net score
    total = pos_count + neg_count
    if total == 0:
        return "Neutral"
    
    score = (pos_count - neg_count) / total
    
    # Define thresholds for classification
    if score > 0.1:
        return "Positive"
    elif score < -0.1:
        return "Negative"
    else:
        return "Neutral"


df_filtered['label'] = df_filtered[mda_column].apply(financial_labels)


print(df_filtered[['Company_name', 'Label']])

                                company_name     label
0     TROPICAL SPORTSWEAR INTERNATIONAL CORP  Positive
2                             DATAWATCH CORP  Negative
3            BEI MEDICAL SYSTEMS CO INC /DE/  Negative
4                           GALEY & LORD INC  Positive
8                                  MEDIQ INC  Positive
...                                      ...       ...
2986                     HELLER FUNDING CORP  Negative
2992         FIRST SECURITYFED FINANCIAL INC   Neutral
2994                          EQUITY ONE INC  Positive
2995              CAPITAL SENIOR LIVING CORP  Positive
2997      STARTEC GLOBAL COMMUNICATIONS CORP  Positive

[1860 rows x 2 columns]


In [32]:
import re

def clean_financial_text(text):
    # Cleaning the text by removing HTML tags, punctuations, extra spaces etc.
    text = str(text).lower()
    
    text = re.sub(r'<[^>]+>', ' ', text)
    
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    words = text.split()
    cleaned_text = " ".join(words)
    
    return cleaned_text

# Applying the text cleaning function to MD&A column
mda_column = 'Management’s Discussion and Analysis of Financial Condition and Results of Operations'
df_filtered['cleaned_MD&A'] = df_filtered[mda_column].apply(clean_financial_text)

print("\nORIGINAL TEXT SNIPPET (First 200 chars):")
print(df_filtered[mda_column].iloc[0][:200])

print("\nCLEANED TEXT SNIPPET (First 200 chars):")
print(df_filtered['cleaned_MD&A'].iloc[0][:200])


ORIGINAL TEXT SNIPPET (First 200 chars):
General           The Company manages the production of substantially all of its products utilizing  Company-owned  facilities  in Tampa,  Florida and El Paso,  Texas and independent assembly manufact

CLEANED TEXT SNIPPET (First 200 chars):
general the company manages the production of substantially all of its products utilizing company owned facilities in tampa florida and el paso texas and independent assembly manufacturers located in 


In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initializing the TF-IDF Vectorizer limiting to 1000 words
vectorizer = TfidfVectorizer(max_features=1000)

# Fit and transform the cleaned text into a numeric matrix
X_tfidf = vectorizer.fit_transform(df_filtered['cleaned_MD&A'])

# Convert the numeric matrix into readable format
X_matrix = X_tfidf.toarray()


print("\nFirst 10 numeric feature values for the first company:")
print(X_matrix[0][:10])


First 10 numeric feature values for the first company:
[0.04206528 0.02518816 0.00189789 0.00656833 0.         0.00215191
 0.00396651 0.0024038  0.         0.00615675]


In [34]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from catboost import CatBoostClassifier

# Map labels to numeric codes
label_map = {'Positive': 1, 'Negative': 0, 'Neutral': 2}
y = df_filtered['label'].map(label_map).values

# Extracting feature matrix from Stage 2 and 80/20 split rule.
X_matrix = X_tfidf.toarray()

X_train, X_test, y_train, y_test = train_test_split(X_matrix, y, test_size=0.2, random_state=42)

# Initializing the models
xgb_model = XGBClassifier(eval_metric='logloss')
ada_model = AdaBoostClassifier(random_state=42)
cat_model = CatBoostClassifier(verbose=0)

# Training the models
xgb_model.fit(X_train, y_train)
ada_model.fit(X_train, y_train)
cat_model.fit(X_train, y_train)

print("\nTraining and Initializing of the models is complete ")


Training and Initializing of the models is complete 


In [35]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

models = {
    "XGBoost": xgb_model,
    "AdaBoost": ada_model,
    "CatBoost": cat_model
}

print("MODEL EVALUATION REPORT")

# Loop through each model and calculate required metrics
for name, model in models.items():
    # Predicting values on test set 
    y_pred = model.predict(X_test)
    
    # Calculate given metrics 
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"\n*** {name} ***")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)


MODEL EVALUATION REPORT

*** XGBoost ***
Accuracy:  0.8306
Precision: 0.8294
Recall:    0.8293
F1-Score:  0.8290
Confusion Matrix:
[[ 93   6  14]
 [  8 118   7]
 [ 13  15  98]]

*** AdaBoost ***
Accuracy:  0.7500
Precision: 0.7620
Recall:    0.7459
F1-Score:  0.7490
Confusion Matrix:
[[ 74   6  33]
 [  6 106  21]
 [ 12  15  99]]

*** CatBoost ***
Accuracy:  0.8495
Precision: 0.8533
Recall:    0.8502
F1-Score:  0.8478
Confusion Matrix:
[[102   3   8]
 [ 10 121   2]
 [ 12  21  93]]
